In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS retailpulse.silver;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS retailpulse.gold;

## Loading Bronze Delta Live Tables

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


customers_bronze = spark.table("retailpulse.bronze.customers")
sales_bronze = spark.table("retailpulse.bronze.sales")
products_bronze = spark.table("retailpulse.bronze.products")
stores_bronze = spark.table("retailpulse.bronze.stores")

In [0]:
customers_bronze.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- registration_date: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _ingestion_batch_id: string (nullable = true)



In [0]:
customers_silver = (
    customers_bronze
    .withColumn("customer_id", F.trim(F.col("customer_id")))
    .withColumn("customer_name", F.trim(F.col("customer_name")))
    .withColumn("city", F.trim(F.col("city")))
    .withColumn("region", F.trim(F.col("region")))
    .withColumn(
        "registration_date",
        F.expr("try_cast(registration_date AS date)")
    )
)

In [0]:
products_silver = (
    products_bronze
    .withColumn("product_id", F.trim(F.col("product_id")))
    .withColumn("product_name", F.trim(F.col("product_name")))
    .withColumn("category", F.trim(F.col("category")))
    .withColumn(
        "list_price",
        F.expr("try_cast(list_price AS DECIMAL(12,2))")
    )
)

In [0]:
stores_silver = (
    stores_bronze
    .withColumn("store_id", F.trim(F.col("store_id")))
    .withColumn("store_name", F.trim(F.col("store_name")))
    .withColumn("city", F.trim(F.col("city")))
    .withColumn("region", F.trim(F.col("region")))
    .withColumn("store_type", F.trim(F.col("store_type")))
)

In [0]:
dimension_checks = [
    (
        customers_silver,
        "customer_id",
        "customers",
        ["customer_name", "city", "region", "registration_date"]
    ),
    (
        products_silver,
        "product_id",
        "products",
        ["product_name", "category", "list_price"]
    ),
    (
        stores_silver,
        "store_id",
        "stores",
        ["store_name", "city", "region", "store_type"]
    )
]

for df, key, name, required_columns in dimension_checks:

    invalid = df.filter(
        F.col(key).isNull() |
        (F.col(key) == "")
    )

    for column in required_columns:
        invalid = invalid.unionByName(
            df.filter(F.col(column).isNull())
        )

    duplicate_keys = (
        df.groupBy(key)
        .count()
        .filter(F.col("count") > 1)
    )

    if invalid.count() > 0 or duplicate_keys.count() > 0:
        raise ValueError(f"Dimension validation failed: {name}")

    print(f"{name}: {df.count()} valid records")

customers: 200 valid records
products: 50 valid records
stores: 10 valid records


In [0]:
storage_account = "retailpulsestorageacc"
container = "retail-data"

base_path = (
    f"abfss://{container}@{storage_account}"
    ".dfs.core.windows.net"
)

silver_path = f"{base_path}/silver"

def save_silver(df, name):
    path = f"{silver_path}/{name}"

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .save(path)
    )

    print(f"Saved Silver dataset: {name}")

In [0]:
save_silver(customers_silver, "customers")
save_silver(products_silver, "products")
save_silver(stores_silver, "stores")

Saved Silver dataset: customers
Saved Silver dataset: products
Saved Silver dataset: stores


In [0]:
sales_typed = (
    sales_bronze
    .withColumn("order_id", F.trim(F.col("order_id")))
    .withColumn("customer_id", F.trim(F.col("customer_id")))
    .withColumn("product_id", F.trim(F.col("product_id")))
    .withColumn("store_id", F.trim(F.col("store_id")))
    .withColumn("status", F.trim(F.col("status")))
    .withColumn(
        "order_date",
        F.expr("try_cast(order_date AS date)")
    )
    .withColumn(
        "quantity",
        F.expr("try_cast(quantity AS INT)")
    )
    .withColumn(
        "unit_price",
        F.expr("try_cast(unit_price AS DECIMAL(12,2))")
    )
    .withColumn(
        "discount",
        F.expr("try_cast(discount AS DECIMAL(12,2))")
    )
    .withColumn(
        "last_modified_date",
        F.expr("try_cast(last_modified_date AS TIMESTAMP)")
    )
)

In [0]:
sales_typed.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: decimal(12,2) (nullable = true)
 |-- discount: decimal(12,2) (nullable = true)
 |-- status: string (nullable = true)
 |-- last_modified_date: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _ingestion_batch_id: string (nullable = true)



In [0]:
customer_keys = (
    customers_silver
    .select(
        F.col("customer_id").alias("valid_customer_id")
    )
)

product_keys = (
    products_silver
    .select(
        F.col("product_id").alias("valid_product_id")
    )
)

store_keys = (
    stores_silver
    .select(
        F.col("store_id").alias("valid_store_id")
    )
)

sales_checked = (
    sales_typed
    .join(
        customer_keys,
        sales_typed.customer_id == customer_keys.valid_customer_id,
        "left"
    )
    .join(
        product_keys,
        F.col("product_id") == F.col("valid_product_id"),
        "left"
    )
    .join(
        store_keys,
        F.col("store_id") == F.col("valid_store_id"),
        "left"
    )
)

### Data Validation

In [0]:
sales_validated = sales_checked.withColumn(
    "rejection_reason",
    F.concat_ws(
        " | ",

        F.when(
            F.col("order_id").isNull() |
            (F.col("order_id") == ""),
            "Missing order ID"
        ),

        F.when(
            F.col("order_date").isNull(),
            "Invalid order date"
        ),

        F.when(
            F.col("quantity").isNull() |
            (F.col("quantity") <= 0),
            "Invalid quantity"
        ),

        F.when(
            F.col("unit_price").isNull() |
            (F.col("unit_price") < 0),
            "Invalid unit price"
        ),

        F.when(
            F.col("discount").isNull() |
            (F.col("discount") < 0) |
            (
                F.col("discount") >
                F.col("quantity") * F.col("unit_price")
            ),
            "Invalid discount"
        ),

        F.when(
            F.col("status").isNull() |
            (~F.col("status").isin(
                "Completed", "Cancelled", "Returned"
            )),
            "Invalid status"
        ),

        F.when(
            F.col("valid_customer_id").isNull(),
            "Unknown or missing customer"
        ),

        F.when(
            F.col("valid_product_id").isNull(),
            "Unknown or missing product"
        ),

        F.when(
            F.col("valid_store_id").isNull(),
            "Unknown or missing store"
        ),

        F.when(
            F.col("last_modified_date").isNull(),
            "Invalid modification timestamp"
        )
    )
)

In [0]:
valid_candidates = sales_validated.filter(
    F.col("rejection_reason") == ""
)

invalid_sales = sales_validated.filter(
    F.col("rejection_reason") != ""
)

In [0]:
print("Valid candidates:", valid_candidates.count())
print("Invalid records:", invalid_sales.count())

Valid candidates: 991
Invalid records: 9


In [0]:
window_spec = (
    Window
    .partitionBy("order_id")
    .orderBy(F.col("last_modified_date").desc())
)

ranked_sales = valid_candidates.withColumn(
    "row_num",
    F.row_number().over(window_spec)
)

In [0]:
sales_current = (
    ranked_sales
    .filter(F.col("row_num") == 1)
    .drop("row_num", "rejection_reason")
)

superseded_sales = (
    ranked_sales
    .filter(F.col("row_num") > 1)
    .drop("row_num")
    .withColumn(
        "rejection_reason",
        F.lit("Superseded by newer order version")
    )
)

In [0]:
rejected_sales = invalid_sales.unionByName(
    superseded_sales
)

print("Current sales:", sales_current.count())
print("Rejected or superseded:", rejected_sales.count())

Current sales: 990
Rejected or superseded: 10


In [0]:
sales_silver = sales_current.withColumn(
    "net_sales",
    F.when(
        F.col("status") == "Completed",
        (
            F.col("quantity") * F.col("unit_price")
            - F.col("discount")
        ).cast("decimal(18,2)")
    ).otherwise(
        F.lit(0).cast("decimal(18,2)")
    )
)

In [0]:
sales_silver = sales_silver.drop(
    "valid_customer_id",
    "valid_product_id",
    "valid_store_id"
)

In [0]:
save_silver(sales_silver, "sales")
save_silver(rejected_sales, "sales_quarantine")

Saved Silver dataset: sales
Saved Silver dataset: sales_quarantine


## Saving Dalta Live Tables

In [0]:
silver_tables = [
    "sales",
    "customers",
    "products",
    "stores",
    "sales_quarantine"
]

for name in silver_tables:
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS retailpulse.silver.{name}
        USING DELTA
        LOCATION '{silver_path}/{name}'
    """)

    print(f"Registered: retailpulse.silver.{name}")

Registered: retailpulse.silver.sales
Registered: retailpulse.silver.customers
Registered: retailpulse.silver.products
Registered: retailpulse.silver.stores
Registered: retailpulse.silver.sales_quarantine


### Accessing Delta Live Tables By using Unity Catalog

In [0]:
%sql
SELECT COUNT(*) AS clean_sales
FROM retailpulse.silver.sales;

clean_sales
990


In [0]:
%sql
SELECT order_id, rejection_reason
FROM retailpulse.silver.sales_quarantine;

order_id,rejection_reason
O00991,Unknown or missing customer
O00992,Invalid quantity | Invalid discount
O00993,Invalid order date
O00994,Unknown or missing product
O00995,Invalid unit price
O00996,Invalid discount
O00997,Invalid status
O00998,Unknown or missing store
O01000,Invalid quantity
O00016,Superseded by newer order version


In [0]:
%sql
SELECT
    status,
    COUNT(*) AS orders,
    SUM(net_sales) AS revenue
FROM retailpulse.silver.sales
GROUP BY status;

status,orders,revenue
Completed,841,4372757.00
Cancelled,103,0.00
Returned,46,0.00


In [0]:
bronze_count = sales_bronze.count()
silver_count = sales_silver.count()
quarantine_count = rejected_sales.count()

assert bronze_count == silver_count + quarantine_count

print("Reconciliation passed")

Reconciliation passed
